In [ ]:
%pip install pandas yfinance fredapi

In [ ]:
import pandas as pd
import yfinance as yf
from fredapi import Fred

In [ ]:
FRED_API_KEY = ""
fred = Fred(api_key=FRED_API_KEY)

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/Project/dataset/marco_and_industry/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def get_fred_data(fred_api_key, series_dict, start_date="2015-01-01"):

    fred = Fred(api_key=fred_api_key)
    data = {}

    for name, series_id in series_dict.items():
        try:
            series = fred.get_series(series_id, start_date=start_date)
            data[name] = series
        except Exception as e:
            print(f"⚠️ 其他錯誤 {name}（{series_id}）：{e}")

    df = pd.DataFrame(data)
    df.index.name = "Date"
    return df


In [ ]:
fred_series = {
    # 通膨與利率
    "CPI": "CPIAUCSL",                          # 消費者物價指數
    "Core_CPI": "CPILFESL",                     # 核心CPI（不含食物與能源）
    "PPI": "PPIACO",                            # 生產者物價指數（總體）
    "FedFundsRate": "FEDFUNDS",                 # 聯邦基金利率

    # 經濟成長與就業
    "GDP_Growth": "A191RL1Q225SBEA",            # GDP 年增長率
    "Unemployment": "UNRATE",                   # 失業率
    "Industrial_Production_Index": "INDPRO",    # 工業生產指數
    "Retail_Sales_MoM": "RSXFS",                # 零售銷售月增率
    "Initial_Jobless_Claims": "ICSA",           # 初次申請失業救濟金人數
    "Inventory_Sales_Ratio": "ISRATIO",         # 商業庫存銷售比

    # 債券殖利率
    "10Y_Treasury_Yield": "GS10",               # 10年期公債殖利率
    "2Y_Treasury_Yield": "GS2",                 # 2年期公債殖利率

    # 貨幣供給與壓力
    "M2_Money_Supply": "M2SL",                  # M2 貨幣供給量
    "Financial_Stress_Index": "STLFSI4",        # 金融壓力指數
    "High_Yield_Spread": "BAMLH0A0HYM2",        # 高收益債信用利差

    # 匯率與國際資金
    "USD_Index": "DTWEXBGS",                    # 美元指數（貿易加權）
    "USDJPY": "DEXJPUS",                        # 美元對日圓
    "USDCNY": "DEXCHUS",                        # 美元對人民幣

    # 商品價格
    "Crude_Oil_Price": "DCOILWTICO",            # 原油價格（西德州）
    "Gold_Price": "IR14270",           # 黃金價格（倫敦上午定盤）

    # 綜合領先指標
    "Leading_Index": "USSLIND",                 # 美國領先經濟指標

    # 產業相關
     "PPI_Electronic_Components": "PCU33443344",      # 半導體成本
    "Total_Vehicle_Sales": "TOTALSA",                # Tesla 關聯
    "Durable_Goods_Orders": "DGORDER",               # Apple、電動車
    "Retail_Sales_Ex_Auto": "RSXFS",                 # Amazon 關聯
    "Nonresidential_Investment": "PNFI"              # B2B 廣告預算 proxy
}

In [ ]:
# === 2) 轉換規則：logdiff / diff_pp / log1p_diff ===
RULES = {
    # logdiff：嚴格正值的指數/價格/匯率/金額水位
    "CPI": "logdiff",
    "Core_CPI": "logdiff",
    "PPI": "logdiff",
    "Industrial_Production_Index": "logdiff",
    "Retail_Sales_MoM": "logdiff",
    "Inventory_Sales_Ratio": "logdiff",
    "M2_Money_Supply": "logdiff",
    "USD_Index": "logdiff",
    "USDJPY": "logdiff",
    "USDCNY": "logdiff",
    "Crude_Oil_Price": "logdiff",
    "Gold_Price": "logdiff",
    "PPI_Electronic_Components": "logdiff",
    "Total_Vehicle_Sales": "logdiff",
    "Durable_Goods_Orders": "logdiff",
    "Retail_Sales_Ex_Auto": "logdiff",
    "Nonresidential_Investment": "logdiff",

    "FedFundsRate": "diff_pp",
    "GDP_Growth": "diff_pp",
    "Unemployment": "diff_pp",
    "10Y_Treasury_Yield": "diff_pp",
    "2Y_Treasury_Yield": "diff_pp",
    "Financial_Stress_Index": "diff_pp",
    "High_Yield_Spread": "diff_pp",
    "Leading_Index": "diff_pp",

    # 允許 0 的件數類
    "Initial_Jobless_Claims": "log1p_diff",
}

# 指定 diff_pp 的縮放方式：'percent'（值像 3.5 代表 3.5%）、'fraction'（0.035 代表 3.5%）、'none'（純點差）
DIFF_PP_SCALE = {
    "FedFundsRate": "percent",
    "GDP_Growth": "percent",
    "Unemployment": "percent",
    "10Y_Treasury_Yield": "percent",
    "2Y_Treasury_Yield": "percent",
    "High_Yield_Spread": "percent",
    "Financial_Stress_Index": "none",
    "Leading_Index": "none",
}

In [ ]:
# 抓取 FRED 資料
fred_df = get_fred_data(FRED_API_KEY, fred_series, start_date="2014-12-01")
fred_df = fred_df[fred_df.index >= "2014-12-01"]
fred_df.tail()

In [ ]:
def to_logdiff(s):
    # 先移除 NaN，再計算 logdiff
    return np.log(s.replace(0, np.nan).dropna()).diff()

def to_log1p_diff(s):
    # 先移除 NaN，再計算 log1p_diff
    return np.log1p(s.dropna()).diff()

def to_diff_pp(s, scale="percent"):
    # 先移除 NaN，再計算 diff
    d = s.dropna().diff()
    if scale == "fraction":
        d = d * 100.0
    return d

In [ ]:
processed_df = fred_df.copy()

In [ ]:
# 建立函數對應字典
func_map = {
    'logdiff': to_logdiff,
    'diff_pp': to_diff_pp,
    'log1p_diff': to_log1p_diff
}

for col, rule in RULES.items():
    if col in processed_df.columns:
        s = pd.to_numeric(processed_df[col], errors="coerce")
        func = func_map.get(rule)

        if func:
            new_col_name = f"{col}_{rule}"
            if rule == 'diff_pp':
                scale = DIFF_PP_SCALE.get(col, "percent")
                processed_df[new_col_name] = func(s, scale=scale)
            else:
                processed_df[new_col_name] = func(s)

print(processed_df)

In [ ]:
processed_df = processed_df.ffill().bfill()

In [ ]:
processed_df.index = pd.to_datetime(processed_df.index)
processed_df = processed_df[processed_df.index >= "2011-01-01"]
processed_df.filter(regex=r"(_pct_chg|_logdiff|_diff)$").to_csv(f"{DRIVE_PATH}/marco_dataset_change2125.csv")